# 03. Надёжный HTTP-клиент

Timeout, ограниченный retry и различение постоянных/временных ошибок.

Стек: Python standard library. Выполняйте ячейки сверху вниз.

In [ ]:
import json, time, urllib.request, urllib.error

def post_json(url, payload, attempts=3, timeout=10):
    data=json.dumps(payload).encode()
    for attempt in range(attempts):
        try:
            req=urllib.request.Request(url,data=data,headers={'Content-Type':'application/json'})
            with urllib.request.urlopen(req,timeout=timeout) as r:
                return json.load(r)
        except urllib.error.HTTPError as e:
            if e.code < 500 and e.code != 429: raise
        except urllib.error.URLError:
            pass
        if attempt+1 < attempts: time.sleep(2**attempt)
    raise RuntimeError('Сервис недоступен после ограниченного числа попыток')

Важно: повтор безопасен, только если операция идемпотентна или имеет уникальный ключ.